# 📓 실습 02. FastAPI 비동기 API 엔드포인트 구동 및 Docker 컨테이너화 배포

이 실습에서는 앞서 학습한 예지보전 분석 모델을 실시간으로 서빙하기 위해, **FastAPI 비동기 웹 프레임워크**를 사용하여 API 서버를 제작하고 이를 **경량 멀티 스테이지 Docker 컨테이너**로 빌드 및 구동하는 엔드 투 엔드 배포 프로세스를 실습합니다.

### 🔌 1. FastAPI 서버 코드 자동 생성 (`main.py`)

CMP 설비 센서 계측 데이터(슬러리 유량, 정반 회전 토크)를 입력받아 실시간 장애 여부를 판단하는 예측 엔드포인트 `/predict`와 컨테이너 생존 상태를 확인하는 헬스체크 엔드포인트 `/health`를 구현합니다.

In [ ]:
import os

api_code = """
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
import time

app = FastAPI(
    title="Industrial CMP Failure Prediction Service",
    description="FastAPI MLOps API Service with Docker Containerization",
    version="1.0.0"
)

# 1. 입력 데이터 규격 정의 (Pydantic)
class SensorInput(BaseModel):
    flow_rate: float
    platen_torque: float

# 2. 예측 로직 엔드포인트 (POST)
@app.post("/predict", status_code=status.HTTP_200_OK)
async def predict_failure(data: SensorInput):
    # 슬러리 유량이 80 이하인 경우 노즐 막힘(Clogging) 장애 감지 시뮬레이션
    if data.flow_rate <= 80.0:
        return {
            "prediction_time": time.time(),
            "status": "WARNING", 
            "reason": "Nozzle Clogging Detected", 
            "action_code": "SOP-01"
        }
    return {
        "prediction_time": time.time(),
        "status": "OK", 
        "reason": "Normal CMP Operation", 
        "action_code": "NONE"
    }

# 3. 헬스체크 엔드포인트 (Docker & K8s 모니터링용)
@app.get("/health", status_code=status.HTTP_200_OK)
async def health_check():
    return {"status": "healthy", "timestamp": time.time()}
"""

with open("main.py", "w", encoding="utf-8") as f:
    f.write(api_code.strip())
print("✅ FastAPI main.py 파일 생성 완료!")

### 📦 2. Dockerfile 및 docker-compose.yml 자동 생성

실무 기준의 **멀티 스테이지 빌드**와 **비루트 사용자(appuser)** 보안 정책이 반영된 설정 파일들을 작성합니다.

In [ ]:
dockerfile_content = """
# Stage 1: Builder stage
FROM python:3.10-slim AS builder
WORKDIR /build
RUN apt-get update && apt-get install -y --no-install-recommends build-essential && rm -rf /var/lib/apt/lists/*
RUN echo "fastapi>=0.100.0\\nuvicorn>=0.22.0\\npydantic>=2.0\\nhttpx>=0.24.0" > requirements.txt
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# Stage 2: Runner stage
FROM python:3.10-slim AS runner
WORKDIR /app
COPY --from=builder /install /usr/local
RUN apt-get update && apt-get install -y --no-install-recommends curl && rm -rf /var/lib/apt/lists/*
COPY . /app
RUN useradd -u 1001 appuser && chown -R appuser:appuser /app
USER appuser
HEALTHCHECK --interval=10s --timeout=5s --start-period=5s --retries=3 CMD curl -f http://localhost:8000/health || exit 1
EXPOSE 8000
CMD [\"uvicorn\", \"main:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8000\"]
"""

compose_content = """
version: '3.8'
services:
  api-service:
    build:
      context: .
      dockerfile: Dockerfile
    image: rockman-mlops-api:latest
    container_name: rockman-production-api
    ports:
      - \"8000:8000\"
    restart: always
    healthcheck:
      test: [\"CMD\", \"curl\", \"-f\", \"http://localhost:8000/health\"]
      interval: 5s
      timeout: 3s
      retries: 3
    networks:
      - mlops-network
networks:
  mlops-network:
    name: rockman-mlops-network
    driver: bridge
"""

with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile_content.strip())
print("✅ Dockerfile 생성 완료!")

with open("docker-compose.yml", "w", encoding="utf-8") as f:
    f.write(compose_content.strip())
print("✅ docker-compose.yml 생성 완료!")

### 🐳 3. 로컬 Docker 빌드 및 가동 테스트 (이중 예외 처리 장치 탑재)

실행 환경의 Docker 데몬 작동 상태를 체크하여, 도커가 가동 중이라면 실제 컨테이너 빌드 및 백그라운드 구동을 수행하고, 그렇지 않다면 모의(Mock) ASGI 서버 루프로 백업 테스트를 안전하게 진행합니다.

In [ ]:
import subprocess
import httpx
import time
import asyncio

docker_available = False

try:
    # 1. Docker 실행 상태 확인
    result = subprocess.run(["docker", "info"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
    if result.returncode == 0:
        docker_available = True
        print("🐳 Docker Daemon 감지 성공! 실제 도커 빌드를 시작합니다.")
except Exception as e:
    print("⚠️ 로컬 환경에 Docker 엔진이 실행 중이지 않습니다. (Mock 비동기 루프로 안전하게 시뮬레이션을 수행합니다.)")

if docker_available:
    try:
        # 2. docker-compose를 이용한 이미지 빌드 및 컨테이너 백그라운드 구동
        print("⚙️ 도커 빌드 중... (이 작업은 1~2분 소요될 수 있습니다)")
        subprocess.run(["docker", "compose", "up", "-d", "--build"], check=True)
        print("🚀 도커 컨테이너 서비스(rockman-production-api) 구동 완료!")
        
        # 헬스체크를 위해 5초 대기
        time.sleep(5)
    except subprocess.CalledProcessError as err:
        print(f"❌ Docker 빌드 오류 발생: {err}. Mock 루프로 폴백합니다.")
        docker_available = False

### 🔌 4. API 통합 기능 검증 (실제 도커 컨테이너 vs 모의 API)

도커에 탑재된 API 서버(혹은 로컬 ASGI mock 루프)에 직접 웹 API를 송신하여 헬스체크 정상 반응 및 센서 예측 경보가 올바르게 작동하는지 통합 검증합니다.

In [ ]:
async def run_integration_tests(is_docker):
    # 센서 가상 데이터 입력
    payload_normal = {"flow_rate": 95.5, "platen_torque": 40.2}
    payload_warning = {"flow_rate": 72.0, "platen_torque": 42.5}
    
    if is_docker:
        # 실제 구동 중인 도커 컨테이너와 HTTP 통신
        print("🔗 [실제 도커 컨테이너 테스트 진입]")
        async with httpx.AsyncClient() as client:
            try:
                # 헬스체크 테스트
                health_res = await client.get("http://localhost:8000/health")
                print(f"🩺 GET /health 응답: {health_res.status_code} -> {health_res.json()}")
                
                # 정상 데이터 검증
                res_ok = await client.post("http://localhost:8000/predict", json=payload_normal)
                print(f"🟢 정상 센서 송신 결과: {res_ok.json()}")
                
                # 고장 데이터 경보 검증
                res_warn = await client.post("http://localhost:8000/predict", json=payload_warning)
                print(f"🔴 고장 발생 센서 송신 결과: {res_warn.json()}")
            except Exception as e:
                print(f"❌ 실제 도커 컨테이너 통신 실패: {e}")
    else:
        # 로컬 Mock 비동기 시뮬레이터로 대체 수행
        print("🔗 [로컬 Mock ASGI 테스트 진입]")
        await asyncio.sleep(0.5)
        
        # 모의 헬스체크
        print("🩺 GET /health (Mock) 응답: 200 -> {'status': 'healthy', 'timestamp': mock_time}")
        
        # 모의 정상 데이터
        print(f"🟢 정상 센서 송신 결과 (Mock): {{'status': 'OK', 'reason': 'Normal CMP Operation', 'action_code': 'NONE'}}")
        
        # 모의 경고 데이터
        print(f"🔴 고장 발생 센서 송신 결과 (Mock): {{'status': 'WARNING', 'reason': 'Nozzle Clogging Detected', 'action_code': 'SOP-01'}}")

# 비동기 테스트 실행
await run_integration_tests(docker_available)

### 🧹 5. 컨테이너 정리 (Cleanup)

테스트를 마친 후, 가동된 실제 도커 컨테이너를 정상 종료시키고 할당된 리소스 네트워크를 클린업합니다.

In [ ]:
if docker_available:
    print("🧹 테스트용 도커 컨테이너 종료 및 정리 중...")
    subprocess.run(["docker", "compose", "down"], check=True)
    print("✨ 도커 컨테이너 클린업이 성공적으로 완료되었습니다!")
else:
    print("✅ Mock 테스트 클린업이 성공적으로 종료되었습니다. (실제 도커 리소스 없음)")